In [1]:
import sys
sys.path.insert(1, '/home/bricej/MyPythonLibrary/StageM2_IVT/library/')
import domain
from domain import *
from climbas import *
import numpy as np
import geopandas as gpd
import xarray as xr
import pandas as pd
from netCDF4 import Dataset
import dask
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

# HIMAP grouped regions

In [2]:
# Grouped regions
region_groups = {
        "Tien Shan": [
            "Eastern Tien Shan", "Dzhungarsky Alatau",
            "Northern/Western Tien Shan", "Central Tien Shan"
        ],
        "Pamir Alay": ["Pamir Alay"],
        "Pamir": ["Western Pamir"],
        "Hindu Kush": ["Eastern Hindu Kush"],
        "Karakoram": ["Karakoram"],
        "Kunlun": ["Eastern Pamir", "Western Kunlun Shan"],
        "Spiti Lahaul": ["Western Himalaya"],
        "Central Himalaya": ["Central Himalaya"],
        "Bhutan": ["Eastern Himalaya"],
        "Nyainqentangla": ["Nyainqentanglha"],
        "Inner TP": [
            "Gangdise Mountains", "Tibetan Interior Mountains",
            "Eastern Kunlun Shan", "Altun Shan", "Hengduan Shan",
            "Tanggula Shan", "Eastern Tibetan Mountains", "Qilian Shan"
        ]
    }

# Data dh/dt

In [3]:
dh = pd.read_csv('/bettik/PROJECTS/pr-regional-climate/bricej/dh_HIMAP_rates.csv')
dh.head()

,subreg,period,area,dhdt,err_dhdt,dvoldt,err_dvoldt,dmdt,err_dmdt,perc_area_meas,perc_area_res,valid_obs,valid_obs_py,area_nodata
0,Eastern Hindu Kush,2000-01-01_2001-01-01,2.934299e+09,-0.038,0.620,-110367590.0,1.818272e+09,-0.0938,1.5455,0.937,0.945,0.69,0.52,13130000.0
1,Eastern Hindu Kush,2000-01-01_2002-01-01,2.934299e+09,-0.085,0.304,-249549614.0,8.908624e+08,-0.2121,0.7574,0.937,0.945,3.76,1.47,13130000.0
2,Eastern Hindu Kush,2000-01-01_2004-01-01,2.934299e+09,-0.095,0.154,-277944533.0,4.530579e+08,-0.2363,0.3855,0.937,0.945,9.32,3.21,13130000.0
3,Eastern Hindu Kush,2000-01-01_2005-01-01,2.934299e+09,-0.089,0.130,-262457364.0,3.804931e+08,-0.2231,0.3238,0.937,0.945,12.71,3.99,13130000.0
4,Eastern Hindu Kush,2000-01-01_2010-01-01,2.934299e+09,-0.116,0.069,-339397247.0,2.012469e+08,-0.2885,0.1723,0.937,0.945,22.98,7.60,13130000.0


In [4]:
# select only the longest period between 2000 and 2019
dh_period = dh.loc[dh["period"] == "2000-01-01_2020-01-01",["subreg", "dhdt", "err_dhdt"]].reset_index(drop=True)
dh_period = dh_period.rename(columns={"subreg": "region"})
dh_period.head()

,region,dhdt,err_dhdt
0,Eastern Hindu Kush,-0.193,0.035
1,Western Himalaya,-0.383,0.043
2,Eastern Himalaya,-0.538,0.051
3,Central Himalaya,-0.477,0.040
4,Karakoram,-0.073,0.030


In [5]:
# Select HIMAP grouped regions

results = [] # pour stocker les résultats des HIMAP grouped regions

for group_name, subregions in region_groups.items():
    
    # Sélection des lignes correspondantes aux sous-regions
    subset = dh_period[dh_period["region"].isin(subregions)]
    
    if len(subset) > 0: # Vérifier qu'il y a des données
        mean_dhdt = subset["dhdt"].mean()
        mean_err = np.sqrt((subset["err_dhdt"]**2).sum()) / len(subset) # erreur avec la formule des dérivées partielles. 
    else:
        mean_dhdt = None
        mean_err = None
    
    results.append({
        "region": group_name,
        "dhdt": mean_dhdt,
        "err_dhdt": mean_err
    })

# Nouveau DataFrame
dh_period_grouped = pd.DataFrame(results)
dh_period_grouped.head()

,region,dhdt,err_dhdt
0,Tien Shan,-0.45875,0.029676
1,Pamir Alay,-0.12200,0.038000
2,Pamir,-0.13800,0.033000
3,Hindu Kush,-0.19300,0.035000
4,Karakoram,-0.07300,0.030000


In [6]:
# Saving file

path = '/bettik/PROJECTS/pr-regional-climate/bricej/paper/'
filename = f"dhdt_2000-2019_HIMAPgrouped_regions.csv"

dh_period_grouped.to_csv(path + filename, index=False)

print(f'Saved File : {path}{filename}')

Saved File : /bettik/PROJECTS/pr-regional-climate/bricej/paper/dhdt_2000-2019_HIMAPgrouped_regions.csv


In [7]:
# verifying the data saved
ds = pd.read_csv('/bettik/PROJECTS/pr-regional-climate/bricej/paper/dhdt_2000-2019_HIMAPgrouped_regions.csv')
ds.head()

,region,dhdt,err_dhdt
0,Tien Shan,-0.45875,0.029676
1,Pamir Alay,-0.12200,0.038000
2,Pamir,-0.13800,0.033000
3,Hindu Kush,-0.19300,0.035000
4,Karakoram,-0.07300,0.030000
